# 23. 요금 인상의 수요 탄력성 정밀 측정

## 분석 배경 및 목적

가격 탄력성(price elasticity of demand)은 교통 요금 정책의 가장 근본적인 의사결정 변수이다. 요금 인상이 수요를 얼마나 감소시키고, 총 수입에 어떤 영향을 미치며, 수요가 원래 수준으로 회복되기까지 얼마나 걸리는지를 정량적으로 측정하는 것은 요금 정책의 사후 평가와 미래 인상 시뮬레이션에 핵심적이다.

본 분석의 방법론적 기반은 다음 연구들에 근거한다:
- **Angrist & Pischke (2009)**, *Mostly Harmless Econometrics*: 준실험적 인과추론(quasi-experimental causal inference)의 표준 교과서로, 단절적 시계열(Interrupted Time Series, ITS) 설계와 DID(Difference-in-Differences) 방법론의 이론적 토대를 제공한다. 본 분석은 별도의 통제군(다른 도시)이 없으므로, 엄밀한 DID보다는 **ITS 설계에 요일/날씨/공휴일 통제**를 추가한 형태로 구성된다.
- **Buchholz et al. (2023)**, NBER *The Demand for Mobility*: 택시/라이드헤일링 시장에서의 **가격 탄력성 추정** 방법론을 체계화하여, 수요 함수의 구간별 이질성(heterogeneity)과 시간에 따른 동적 반응을 강조하였다.

서울 택시 요금은 분석 기간 내 2회 인상되었으며, 두 인상 모두 약 26%의 유사한 인상률을 가진다. 이를 동일 프레임워크로 분석하여 시장 환경별 탄력성 차이를 비교한다.

| 인상 | 시점 | 기본요금 | 인상률 |
|------|------|----------|--------|
| 1차 | 2018-10-01 | 3,000 -> 3,800원 | +26.7% |
| 2차 | 2023-02-01 | 3,800 -> 4,800원 | +26.3% |

**분석 내용:**
- 각 인상 전후 60일간 일별 수요 비교
- 요일/시간대/날씨 통제 후 순수 요금 효과 추출
- 단거리 vs 장거리 탄력성 차이 (PAY_AMT 구간별)
- 수요 회복 기간 산출

**15번(Causal Impact)과의 차별점:** 본 분석은 요금 인상에 특화된 ITS/DID 설계로, Bayesian structural time series 대신 **준실험적 인과추론**에 초점을 맞춘다. 2개 인상 사례를 동일 프레임워크로 비교하여 시장 환경별 탄력성 차이를 분석한다.

| 데이터 | 용도 |
|--------|------|
| DC_TBYXD012 | 일별 수요, 매출, 거리 |
| calendar | 요일/공휴일 통제 |
| weather_daily | 날씨 통제 |
| taxi_events | 인상 시점 확인 |

In [ ]:
# 필요 라이브러리 설치
# !pip install pandas numpy matplotlib seaborn statsmodels psutil

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import platform
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# === 메모리 유틸 ===
import gc, psutil, os

def mem_usage(tag=''):
    gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'[MEM {tag}] {gb:.2f} GB')

CHUNK_SIZE = 1_000_000
mem_usage('start')

In [ ]:
# === 경로 설정 ===
D012_PATH     = './DC_TBYXD012.csv'
EXT_DIR       = './external_data'
CALENDAR      = f'{EXT_DIR}/calendar_2018_2026.csv'
WEATHER_DAILY = f'{EXT_DIR}/weather_asos_daily_seoul_2018_2026.csv'
EVENTS        = f'{EXT_DIR}/taxi_events_timeline.csv'

# 요금 인상 시점 정의
HIKE_1 = pd.Timestamp('2018-10-01')  # 3000 -> 3800 (+26.7%)
HIKE_2 = pd.Timestamp('2023-02-01')  # 3800 -> 4800 (+26.3%)
WINDOW = 60  # 전후 60일

## 0. 외부 데이터 로드

In [ ]:
cal = pd.read_csv(CALENDAR, encoding='utf-8', parse_dates=['date'])
weather = pd.read_csv(WEATHER_DAILY, encoding='utf-8', parse_dates=['date'])
events = pd.read_csv(EVENTS, encoding='utf-8', parse_dates=['date'])

# 날씨 파생 변수
weather['rainfall'] = weather['rainfall'].fillna(0)
weather['is_rainy'] = (weather['rainfall'] > 0).astype(int)
weather['is_heavy_rain'] = (weather['rainfall'] >= 30).astype(int)

print(f'캘린더: {len(cal):,} | 날씨: {len(weather):,} | 이벤트: {len(events):,}')

# 요금 인상 이벤트
events[events['event'].str.contains('요금')]

## 1. 청크 집계: 인상 전후 기간 일별 수요

분석 대상 기간:
- 1차: 2018-08-02 ~ 2018-11-29 (전후 60일)
- 2차: 2022-12-03 ~ 2023-04-01 (전후 60일)

PAY_AMT 구간으로 단거리/장거리 구분:
- 단거리: 기본요금 이하
- 중거리: 기본요금 ~ 2배
- 장거리: 기본요금 2배 초과

In [ ]:
# 분석 대상 기간 설정
h1_start, h1_end = HIKE_1 - pd.Timedelta(days=WINDOW), HIKE_1 + pd.Timedelta(days=WINDOW)
h2_start, h2_end = HIKE_2 - pd.Timedelta(days=WINDOW), HIKE_2 + pd.Timedelta(days=WINDOW)

print(f'1차 분석 기간: {h1_start.date()} ~ {h1_end.date()}')
print(f'2차 분석 기간: {h2_start.date()} ~ {h2_end.date()}')

usecols = ['RIDE_DTIME', 'PAY_AMT', 'RIDE_DIST']
dtypes = {'RIDE_DTIME': str, 'PAY_AMT': 'float32', 'RIDE_DIST': 'float32'}

# 일별 집계 (전체 + 요금 구간별)
daily_parts = []
daily_dist_parts = []

total_rows = 0
for i, chunk in enumerate(pd.read_csv(D012_PATH, usecols=usecols, dtype=dtypes, chunksize=CHUNK_SIZE)):
    rd = pd.to_datetime(chunk['RIDE_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    m = rd.notna()
    chunk = chunk[m].copy()
    rd = rd[m]
    chunk['date'] = rd.dt.normalize()
    chunk['hour'] = rd.dt.hour.astype('int8')
    
    # 분석 기간 필터링
    in_h1 = (chunk['date'] >= h1_start) & (chunk['date'] <= h1_end)
    in_h2 = (chunk['date'] >= h2_start) & (chunk['date'] <= h2_end)
    sub = chunk[in_h1 | in_h2].copy()
    
    if len(sub) > 0:
        # 어떤 인상 이벤트에 속하는지
        # h1(2018)·h2(2023) 윈도우는 겹치지 않으므로 날짜로 직접 판정
        sub['hike'] = np.where(sub['date'] <= h1_end, 1, 2)
        sub['post'] = np.where(
            (sub['hike'] == 1) & (sub['date'] >= HIKE_1), 1,
            np.where((sub['hike'] == 2) & (sub['date'] >= HIKE_2), 1, 0)
        )
        
        # 요금 구간 분류 (인상별 기본요금 기준)
        base_fare = np.where(sub['hike'] == 1,
                             np.where(sub['post'] == 0, 3000, 3800),
                             np.where(sub['post'] == 0, 3800, 4800))
        sub['fare_segment'] = np.where(
            sub['PAY_AMT'] <= base_fare, 'short',
            np.where(sub['PAY_AMT'] <= base_fare * 2, 'mid', 'long')
        )
        
        # --- 일별 전체 집계 ---
        d = sub.groupby(['date', 'hike', 'post']).agg(
            trips=('PAY_AMT', 'count'),
            revenue=('PAY_AMT', 'sum'),
            avg_fare=('PAY_AMT', 'mean'),
            avg_dist=('RIDE_DIST', 'mean')
        ).reset_index()
        daily_parts.append(d)
        
        # --- 일별 요금 구간별 집계 ---
        dd = sub.groupby(['date', 'hike', 'post', 'fare_segment']).agg(
            trips=('PAY_AMT', 'count'),
            revenue=('PAY_AMT', 'sum')
        ).reset_index()
        daily_dist_parts.append(dd)
    
    total_rows += len(chunk)
    if (i + 1) % 5 == 0:
        mem_usage(f'chunk {i+1}, rows={total_rows:,}')
    del chunk, rd, sub
    gc.collect()

print(f'총 {total_rows:,}건 처리')
mem_usage('after chunking')

In [ ]:
# 청크 합산
daily = pd.concat(daily_parts, ignore_index=True)
del daily_parts; gc.collect()
daily = daily.groupby(['date', 'hike', 'post']).sum().reset_index()
# avg_fare, avg_dist 재계산
daily['avg_fare'] = daily['revenue'] / daily['trips']

daily_dist = pd.concat(daily_dist_parts, ignore_index=True)
del daily_dist_parts; gc.collect()
daily_dist = daily_dist.groupby(['date', 'hike', 'post', 'fare_segment']).sum().reset_index()

print(f'일별 집계: {len(daily):,}건 | 구간별: {len(daily_dist):,}건')
daily.head()

## 2. 외부 데이터 조인 (캘린더, 날씨)

In [ ]:
# 캘린더 조인
daily = daily.merge(cal[['date', 'day_of_week', 'is_weekend', 'is_holiday', 'is_non_working']],
                    on='date', how='left')

# 날씨 조인
daily = daily.merge(weather[['date', 'avg_temp', 'rainfall', 'is_rainy', 'is_heavy_rain']],
                    on='date', how='left')

# 인상 시점 대비 경과일
daily['days_from_hike'] = daily.apply(
    lambda r: (r['date'] - HIKE_1).days if r['hike'] == 1 else (r['date'] - HIKE_2).days,
    axis=1
)

print(daily[['date', 'hike', 'post', 'trips', 'day_of_week', 'is_rainy', 'days_from_hike']].head(10))

## 3. 인상 전후 수요 추이 시각화

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

for idx, (hike_num, hike_date, label) in enumerate([
    (1, HIKE_1, '1차 인상 (3000->3800, 2018-10)'),
    (2, HIKE_2, '2차 인상 (3800->4800, 2023-02)')
]):
    ax = axes[idx]
    sub = daily[daily['hike'] == hike_num].sort_values('date')
    
    # 전/후 색상 구분
    pre = sub[sub['post'] == 0]
    post = sub[sub['post'] == 1]
    
    ax.plot(pre['days_from_hike'], pre['trips'], 'o-', color='#1976d2', ms=4, lw=1, label='인상 전')
    ax.plot(post['days_from_hike'], post['trips'], 'o-', color='#d32f2f', ms=4, lw=1, label='인상 후')
    ax.axvline(0, color='black', ls='--', lw=2, label='인상일')
    
    # 7일 이동평균
    sub_sorted = sub.sort_values('days_from_hike')
    ma7 = sub_sorted['trips'].rolling(7, center=True).mean()
    ax.plot(sub_sorted['days_from_hike'], ma7, color='#f57c00', lw=2.5, alpha=0.8, label='7일 이동평균')
    
    ax.set_title(f'{label}', fontsize=13)
    ax.set_xlabel('인상 시점 대비 일수')
    ax.set_ylabel('일별 수요 (건)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('요금 인상 전후 일별 수요 추이', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 4. DID 회귀분석: 순수 요금 효과 추출

Angrist & Pischke (2009)의 프레임워크에 따라, 요금 인상의 인과효과를 추정하기 위해 외부 교란 변수를 통제한 회귀 모형을 구성한다.

**모델:** `trips ~ post + hike + post:hike + day_of_week + is_rainy + avg_temp`

- `post`: 인상 후 더미 (0/1) -- 인상의 직접 효과
- `hike`: 인상 차수 (1차/2차) -- 시기 고정효과 (두 시기의 수준 차이 통제)
- `post:hike`: DID 교호작용 -- 2차 인상의 추가 효과
- 통제변수: 요일, 비/폭우, 기온, 비영업일

**이분산 강건 표준오차(HC1)**를 적용하여, 시계열 데이터에서 흔한 이분산 문제에 대한 추론의 robustness를 확보한다.

**방법론 주의:** 두 인상 모두 '처치(treatment)'이고 별도 통제군(다른 도시)이 없으므로 엄밀한 DID가 아니라 **단절적 시계열(ITS) + 두 사건 비교**에 가깝다. `post:hike2` 계수는 '2차 인상이 1차 대비 추가로 준 효과'로 해석해야 하며, 통제군 대비 인과효과로 읽지 말 것. 요일/날씨/공휴일을 통제한 `post` 계수만 수요 변화 추정치로 보고한다.

In [ ]:
# DID 데이터 준비
did = daily.copy()
did['hike_2'] = (did['hike'] == 2).astype(int)  # 2차 인상 더미
did['post_x_hike2'] = did['post'] * did['hike_2']  # DID 교호작용

# 요일 더미 (월요일 기준)
did = pd.get_dummies(did, columns=['day_of_week'], prefix='dow', drop_first=True, dtype=int)

# 회귀분석
controls = [c for c in did.columns if c.startswith('dow_')] + ['is_rainy', 'avg_temp', 'is_non_working']
X_cols = ['post', 'hike_2', 'post_x_hike2'] + controls

X = sm.add_constant(did[X_cols].fillna(0))
y = did['trips']

model = sm.OLS(y, X).fit(cov_type='HC1')  # 이분산 강건 표준오차
print(model.summary())

In [ ]:
# 각 인상 개별 DID
for hike_num, hike_label in [(1, '1차 인상 (2018-10)'), (2, '2차 인상 (2023-02)')]:
    sub = daily[daily['hike'] == hike_num].copy()
    sub = pd.get_dummies(sub, columns=['day_of_week'], prefix='dow', drop_first=True, dtype=int)
    
    ctrl = [c for c in sub.columns if c.startswith('dow_')] + ['is_rainy', 'avg_temp', 'is_non_working']
    X = sm.add_constant(sub[['post'] + ctrl].fillna(0))
    y = sub['trips']
    
    m = sm.OLS(y, X).fit(cov_type='HC1')
    
    pre_mean = sub[sub['post'] == 0]['trips'].mean()
    coef = m.params['post']
    pct_change = coef / pre_mean * 100
    
    print(f'\n=== {hike_label} ===')
    print(f'  인상 전 일평균 수요: {pre_mean:,.0f}건')
    print(f'  인상 후 변화 (통제 후): {coef:+,.0f}건 ({pct_change:+.1f}%)')
    print(f'  p-value: {m.pvalues["post"]:.4f}')
    print(f'  요금 인상률: {26.7 if hike_num == 1 else 26.3}%')
    print(f'  수요 탄력성: {pct_change / (26.7 if hike_num == 1 else 26.3):.3f}')

## 5. 요금 구간별 탄력성 차이 (단거리 vs 장거리)

Buchholz et al. (2023)은 수요 함수의 **구간별 이질성(heterogeneity)**을 강조하며, 가격 수준에 따라 탄력성이 다르다고 보고하였다. 경제 이론적으로:

- **단거리(기본요금 이하):** 도보, 버스, 지하철 등 대체 수단이 다양하므로 탄력성이 높음 (|e| > 1)
- **중거리(기본요금-2배):** 대안이 제한적이나 여전히 대중교통 이용 가능, 중간 탄력성
- **장거리(기본요금 2배 초과):** 택시 외 대안이 거의 없어 비탄력적 (|e| < 1)

요금 구간 분류는 인상 전/후 기본요금을 기준으로 동적으로 설정하여, 인상으로 인한 구간 이동 효과를 반영한다.

In [ ]:
# 구간별 일별 데이터에 캘린더/날씨 조인
daily_dist = daily_dist.merge(
    cal[['date', 'day_of_week', 'is_non_working']], on='date', how='left'
)
daily_dist = daily_dist.merge(
    weather[['date', 'avg_temp', 'rainfall', 'is_rainy']], on='date', how='left'
)

# 구간별 탄력성 계산
elasticity_results = []
for hike_num, hike_label, pct_hike in [(1, '1차', 26.7), (2, '2차', 26.3)]:
    for seg in ['short', 'mid', 'long']:
        sub = daily_dist[(daily_dist['hike'] == hike_num) & (daily_dist['fare_segment'] == seg)].copy()
        if len(sub) < 20:
            continue
        sub = pd.get_dummies(sub, columns=['day_of_week'], prefix='dow', drop_first=True, dtype=int)
        ctrl = [c for c in sub.columns if c.startswith('dow_')] + ['is_rainy', 'avg_temp', 'is_non_working']
        X = sm.add_constant(sub[['post'] + ctrl].fillna(0))
        y = sub['trips']
        
        m = sm.OLS(y, X).fit(cov_type='HC1')
        pre_mean = sub[sub['post'] == 0]['trips'].mean()
        coef = m.params['post']
        pct_change = coef / pre_mean * 100 if pre_mean > 0 else 0
        
        elasticity_results.append({
            'hike': hike_label,
            'segment': seg,
            'pre_mean': pre_mean,
            'post_effect': coef,
            'pct_change': pct_change,
            'elasticity': pct_change / pct_hike,
            'p_value': m.pvalues['post']
        })

elas_df = pd.DataFrame(elasticity_results)
print('=== 요금 구간별 수요 탄력성 ===')
elas_df.round(4)

In [ ]:
# 시각화: 요금 구간별 탄력성 바 차트
seg_labels = {'short': '단거리', 'mid': '중거리', 'long': '장거리'}
elas_df['seg_label'] = elas_df['segment'].map(seg_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, hike_label in enumerate(['1차', '2차']):
    ax = axes[idx]
    sub = elas_df[elas_df['hike'] == hike_label]
    colors = ['#42a5f5', '#66bb6a', '#ef5350']
    bars = ax.bar(sub['seg_label'], sub['elasticity'], color=colors, edgecolor='white', width=0.5)
    
    # 값 표시
    for bar, val, pv in zip(bars, sub['elasticity'], sub['p_value']):
        sig = '***' if pv < 0.01 else '**' if pv < 0.05 else '*' if pv < 0.1 else ''
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.3f}{sig}', ha='center', va='bottom', fontsize=12)
    
    ax.axhline(0, color='black', lw=0.5)
    ax.set_title(f'{hike_label} 인상 - 요금 구간별 수요 탄력성', fontsize=13)
    ax.set_ylabel('수요 탄력성')
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
print('* p<0.1, ** p<0.05, *** p<0.01')

## 6. 수요 회복 기간 산출

가격 탄력성은 **단기(short-run)와 장기(long-run)**로 구분된다. 단기적으로 수요가 급감하더라도, 소비자가 새로운 가격 수준에 적응하면서 수요가 점진적으로 회복되는 **동적 반응 경로(dynamic response path)**가 존재한다.

회복 기간의 측정 방법:
- 7일 이동평균이 인상 전 평균 수준을 최초로 회복하는 시점을 **회복일(recovery day)**로 정의한다.
- 이동평균을 사용하여 요일별 변동(주중/주말 차이)에 의한 노이즈를 제거한다.
- 회복일이 관측 기간(60일) 내에 나타나지 않으면, 해당 인상의 영향이 장기적으로 지속됨을 시사한다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

recovery_results = []
for idx, (hike_num, hike_date, label) in enumerate([
    (1, HIKE_1, '1차 인상 (2018-10)'),
    (2, HIKE_2, '2차 인상 (2023-02)')
]):
    sub = daily[daily['hike'] == hike_num].sort_values('date').copy()
    pre_mean = sub[sub['post'] == 0]['trips'].mean()
    
    # 인상 후 데이터
    post_data = sub[sub['post'] == 1].sort_values('days_from_hike').copy()
    post_data['ma7'] = post_data['trips'].rolling(7, min_periods=1).mean()
    
    # 회복일 찾기 (7일 이동평균이 인상 전 평균 도달)
    recovered = post_data[post_data['ma7'] >= pre_mean]
    recovery_day = recovered['days_from_hike'].iloc[0] if len(recovered) > 0 else None
    
    recovery_results.append({
        'hike': label,
        'pre_mean': pre_mean,
        'recovery_day': recovery_day
    })
    
    # 시각화
    ax = axes[idx]
    ax.plot(post_data['days_from_hike'], post_data['trips'], 'o', color='#90a4ae', ms=3, alpha=0.5)
    ax.plot(post_data['days_from_hike'], post_data['ma7'], color='#d32f2f', lw=2.5, label='7일 이동평균')
    ax.axhline(pre_mean, color='#1976d2', ls='--', lw=2, label=f'인상 전 평균 ({pre_mean:,.0f}건)')
    if recovery_day is not None:
        ax.axvline(recovery_day, color='#388e3c', ls=':', lw=2, label=f'회복일 (D+{recovery_day})')
    ax.set_title(f'{label} - 수요 회복 곡선', fontsize=13)
    ax.set_xlabel('인상 후 경과일')
    ax.set_ylabel('일별 수요 (건)')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

recovery_df = pd.DataFrame(recovery_results)
print('\n=== 수요 회복 기간 ===')
recovery_df

## 7. 2개 인상 비교: 탄력성 요약

In [ ]:
# 두 인상의 탄력성 비교 요약
summary = []
for hike_num, label, pct_hike, base_fare in [
    (1, '1차 (2018-10)', 26.7, '3000->3800'),
    (2, '2차 (2023-02)', 26.3, '3800->4800')
]:
    sub = daily[daily['hike'] == hike_num]
    pre = sub[sub['post'] == 0]
    post = sub[sub['post'] == 1]
    
    pre_mean = pre['trips'].mean()
    post_mean = post['trips'].mean()
    raw_change = (post_mean - pre_mean) / pre_mean * 100
    
    summary.append({
        '인상': label,
        '기본요금': base_fare,
        '인상률(%)': pct_hike,
        '인상전_일평균': f'{pre_mean:,.0f}',
        '인상후_일평균': f'{post_mean:,.0f}',
        '수요변화(%)': f'{raw_change:+.1f}',
        '원시_탄력성': f'{raw_change / pct_hike:.3f}',
        '회복일': recovery_df[recovery_df['hike'].str.contains(label[:2])]['recovery_day'].values[0]
    })

summary_df = pd.DataFrame(summary)
print('=== 요금 인상 탄력성 비교 요약 ===')
summary_df

## 8. 종합 분석 및 결론

### 분석 결과 요약

#### ITS/DID 분석 설계
- 2개의 유사한 요금 인상(각 약 26%)을 동일 프레임워크로 분석
- 요일, 날씨(강수/기온), 공휴일 통제 후 순수 요금 효과 추출
- 이분산 강건 표준오차(HC1) 적용으로 추론의 신뢰성 확보

#### 주요 발견
- **1차 인상 (2018-10)**: 카카오T 본격화 직후, 플랫폼 성장기
  - 요금 인상에도 플랫폼 확산 효과가 수요 감소를 상쇄할 가능성
- **2차 인상 (2023-02)**: 코로나 이후 회복기, 성숙한 플랫폼 시장
  - 26%대 인상에 대한 시장의 반응이 첫 인상과 어떻게 달랐는지

#### 요금 구간별 탄력성
- 경제 이론상 단거리 수요가 더 탄력적 (대체재 이용 가능: 도보, 버스 등)
- 장거리 수요는 비탄력적 (택시 외 대안 부족)
- 실제 데이터에서 이 패턴이 관찰되는지 확인

#### 15번 Causal Impact 분석과의 비교
| 항목 | 15번 (Causal Impact) | 23번 (ITS/DID) |
|------|---------------------|--------------|
| 방법론 | Bayesian structural time series | Interrupted Time Series + DID |
| 대조군 | 합성 대조군 (시계열 예측) | 인상 전 기간 (시간적 대조) |
| 초점 | 개별 이벤트 영향 전반 | 요금 인상에 특화 |
| 장점 | 유연한 시계열 모델링 | 직관적 해석, 구간별 이질적 효과 |
| 본 분석 추가 | - | 2개 인상 비교, 요금구간별 탄력성, 회복 기간 |

### 실무 활용
- **미래 요금 인상 시뮬레이션**: 본 분석에서 추정된 탄력성 계수를 활용하면, 차기 요금 인상 시 예상 수요 감소폭과 총수입 변화를 사전에 시뮬레이션할 수 있다.
- **구간별 차등 인상 전략**: 단거리 수요가 탄력적이라면, 기본요금보다 거리 비례 요금을 인상하는 것이 수요 감소를 최소화하면서 수입을 확보하는 전략이 될 수 있다.
- **회복 기간 기반 정책 타이밍**: 수요 회복에 소요되는 기간을 고려하여, 인상 시점을 수요가 높은 계절(겨울)에 맞추거나 이벤트와 동시에 시행하는 것이 충격을 완화하는 전략이 될 수 있다.
- **인상 한도 설정 근거**: 탄력성이 -1을 넘으면(탄력적) 요금 인상이 오히려 총수입을 감소시키므로, 구간별 탄력성 추정치가 요금 인상 한도의 경제학적 근거를 제공한다.

#### 한계점
- 순수 DID는 병렬적 대조군(다른 도시 등)이 이상적이나, 본 분석은 시간적 전후 비교에 의존
- 1차 인상 시기 플랫폼 효과, 2차 인상 시기 코로나 회복 효과가 혼재
- 60일 윈도우 내 계절 효과가 일부 작용할 가능성

## References

1. Angrist, J. D., & Pischke, J. S. (2009). *Mostly Harmless Econometrics: An Empiricist's Companion*. Princeton University Press.
2. Buchholz, N., Doval, L., Kastl, J., Matejka, F., & Salz, T. (2023). The Demand for Mobility. *NBER Working Paper No. 31466*.
3. Bernal, J. L., Cummins, S., & Gasparrini, A. (2017). Interrupted time series regression for the evaluation of public health interventions: a tutorial. *International Journal of Epidemiology*, 46(1), 348-355.
4. Schaller, B. (2007). Entry Controls in Taxi Regulation: Implications of US and Canadian Experience for Taxi Regulation and Deregulation. *Transport Policy*, 14(6), 490-506.

In [ ]:
mem_usage('final')
print('=== 23_fare_hike_elasticity 분석 완료 ===')